# 作业3：NumPy、Matplotlib 与 pandas

本 Notebook 包含三部分：

1. 半圆定律：Wigner 矩阵与特征值分布；
2. Conway 生命游戏：棋盘验证、演化与绘图；
3. pandas DataFrame：读取、修正、扩展、保存与分组汇总 iris 数据。

> 请先将第一个代码框中的学号修改为自己的学号，再按顺序运行；提交前重启内核并运行全部代码。

## 0. 身份与环境记录

In [ ]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080228
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 一、围绕半圆定律

半圆密度为

\[
f(x)=
\begin{cases}
\dfrac{1}{2\pi}\sqrt{4-x^2}, & |x|\le 2,\\
0, & |x|>2.
\end{cases}
\]

对 \(n\times n\) 的 Wigner 矩阵 \(Z\)：对所有 \(1\le i\le j\le n\)，独立地从均值为 0、方差为 \(1/n\) 的正态分布抽取 \(Z_{ij}\)，再令 \(Z_{ji}=Z_{ij}\)。

### 1.1 半圆密度函数

In [ ]:
def wigner_density(x):
    """返回半圆密度在 x 处的值；x 可以是标量或 NumPy 数组。"""
    x_arr = np.asarray(x)
    values = np.sqrt(np.maximum(0.0, 4.0 - x_arr**2)) / (2 * np.pi)
    result = np.where(np.abs(x_arr) <= 2, values, 0.0)
    return float(result) if np.ndim(result) == 0 else result

In [ ]:
for x in [-3, -2, 0, 1.5, 2, 3, np.float64(1.0)]:
    print(f"f({x}) = {wigner_density(x)}")

print("数组输入:", wigner_density(np.array([-2.0, 0.0, 2.0, 3.0])))

### 1.2 生成 Wigner 矩阵

In [ ]:
def generate_wigner(n):
    """生成 n 阶对称 Wigner 矩阵；n 必须为正整数。"""
    if isinstance(n, (bool, np.bool_)) or not isinstance(n, (int, np.integer)) or n <= 0:
        raise ValueError("n 必须是正整数")

    z = np.random.normal(loc=0.0, scale=1.0 / np.sqrt(n), size=(n, n))
    upper = np.triu(z)
    return upper + np.triu(upper, k=1).T

### 1.3 提取有序特征值

In [ ]:
def get_spectrum(matrix):
    """返回对称矩阵的全部特征值，按非递减顺序排列。"""
    return np.linalg.eigvalsh(matrix)

In [ ]:
W = generate_wigner(4)
print("Wigner 矩阵:\n", W)
print("是否对称:", np.allclose(W, W.T))
print("特征值:", get_spectrum(W))
print("是否有序:", np.all(np.diff(get_spectrum(W)) >= 0))

for bad in [0, -3, 3.5, True]:
    try:
        generate_wigner(bad)
    except Exception as exc:
        print(f"{bad!r} -> {type(exc).__name__}: {exc}")

### 1.4 四个阶数下的直方图与密度曲线

对 \(n=100,200,500,1000\)，各生成一次随机 Wigner 矩阵，用蓝色标准化直方图绘制特征值，并叠加红色半圆密度曲线。

In [ ]:
np.random.seed(0)

n_values = [100, 200, 500, 1000]
xs = np.linspace(-2.5, 2.5, 600)
fig, axes = plt.subplots(4, 1, figsize=(8, 12), sharex=True)

for ax, n in zip(axes, n_values):
    eigenvalues = get_spectrum(generate_wigner(n))
    ax.hist(eigenvalues, bins=60, density=True, color="blue", alpha=0.55,
            label="Normalized histogram")
    ax.plot(xs, wigner_density(xs), color="red", linewidth=2, label="Semicircle density")
    ax.set_title(f"n = {n}")
    ax.set_ylabel("Density")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Eigenvalue")
fig.tight_layout()
plt.show()

**观察结论**

随着 \(n\) 增大，特征值的标准化直方图逐渐接近半圆密度：

- \(n=100,200\) 时边缘和中间仍有明显起伏；
- \(n\approx 500\) 时已经能较好地贴合理想曲线；
- \(n=1000\) 时拟合更平滑、更接近半圆密度。

因此可认为 \(n\) 大约达到 **500** 时半圆密度已经能较好地拟合特征值分布（\(n=1000\) 效果更佳）。Kolmogorov–Smirnov 检验为可选任务，这里未做。

## 二、 Conway 的生命游戏

棋盘为 `m × n` 的 NumPy 二维数组：活细胞为 1，死细胞为 0。每一步同时更新，邻居包括上、下、左、右和四个对角方向，共 8 个，并采用环绕边界。

规则：

- 活细胞邻居少于 2 个：死亡；
- 活细胞邻居为 2 或 3 个：存活；
- 活细胞邻居多于 3 个：死亡；
- 死细胞邻居恰好 3 个：复活；
- 其他死细胞保持死亡。

### 2.1 棋盘合法性检查

In [ ]:
def is_valid_board(board):
    """判断是否为二维且元素仅为 0/1 的 NumPy 数组。"""
    return (
        isinstance(board, np.ndarray)
        and board.ndim == 2
        and bool(np.all(np.isin(board, [0, 1])))
    )

### 2.2 演算一步

In [ ]:
def gol_step(board):
    """按生命游戏规则返回下一步棋盘；输入必须是合法棋盘。"""
    if not is_valid_board(board):
        raise ValueError("board 必须是二维且元素仅为 0/1 的 NumPy 数组")

    neighbors = np.zeros_like(board, dtype=int)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            neighbors += np.roll(np.roll(board, di, axis=0), dj, axis=1)

    alive = board == 1
    survive = alive & ((neighbors == 2) | (neighbors == 3))
    born = (~alive) & (neighbors == 3)
    return np.where(survive | born, 1, 0).astype(int)

### 2.3 绘制棋盘

In [ ]:
def draw_gol_board(board):
    """将棋盘画为 m×n 个方格：活细胞黑、死细胞白。"""
    if not is_valid_board(board):
        raise ValueError("board 必须是二维且元素仅为 0/1 的 NumPy 数组")

    cmap = plt.get_cmap("gray")
    plt.figure(figsize=(5, 5))
    plt.imshow(board, cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.tight_layout()
    plt.show()

### 2.4 合法性测试与滑翔机示例

In [ ]:
print("二维 0/1 数组:", is_valid_board(np.array([[0, 1], [1, 0]])))
print("Python 列表:   ", is_valid_board([[0, 1], [1, 0]]))
print("一维数组:      ", is_valid_board(np.array([0, 1])))
print("含元素 2:      ", is_valid_board(np.array([[0, 2]])))

try:
    gol_step(np.array([[0, 2]]))
except ValueError as exc:
    print("捕获异常:", exc)

In [ ]:
board = np.zeros((8, 8), dtype=int)
for row, col in [(1, 2), (2, 3), (3, 1), (3, 2), (3, 3)]:
    board[row, col] = 1

print("初始棋盘:\n", board)
draw_gol_board(board)

next_board = gol_step(board)
print("一步之后:\n", next_board)
draw_gol_board(next_board)

## 三、使用 pandas DataFrame

本部分读取同目录下的 `iris.csv`（第一行是列标题），完成数据检查、修正、特征扩展、保存和分组汇总。

### 3.1 读取数据并回答基本信息

In [ ]:
import os

csv_path = "iris.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError("请将 iris.csv 放在本 Notebook 同一目录后重新运行")

df = pd.read_csv(csv_path)
print("数据点数量:", len(df))
print("列名:", df.columns.tolist())
print("各列数据类型:")
print(df.dtypes)
print("花卉种类数:", df["species"].nunique())
print("花卉种类:", sorted(df["species"].unique()))

df.head()

**回答**

- 共有 150 个数据点；
- 列名为 `sepal_length`、`sepal_width`、`petal_length`、`petal_width`、`species`；
- 四个测量列都是 `float64`，`species` 为字符串（`object`）；
- 共有 3 种花卉：setosa、versicolor、virginica。

### 3.2 修正第 35、38 行

不计列标题的第 35、38 行对应默认索引 34 和 37。

In [ ]:
columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
df.loc[34, columns] = [4.9, 3.1, 1.5, 0.2, "setosa"]
df.loc[37, columns] = [4.9, 3.6, 1.4, 0.1, "setosa"]

print("修正后的第 35、38 行（索引 34、37）:")
print(df.loc[[34, 37]])

### 3.3 添加两个比值特征

In [ ]:
df["Petal.Ratio"] = df["petal_length"] / df["petal_width"]
df["Sepal.Ratio"] = df["sepal_length"] / df["sepal_width"]

df[["petal_length", "petal_width", "Petal.Ratio",
    "sepal_length", "sepal_width", "Sepal.Ratio"]].head()

### 3.4 保存为 iris_corrected.csv

In [ ]:
output_path = "iris_corrected.csv"
df.to_csv(output_path, index=False)
print("已保存:", os.path.abspath(output_path))

### 3.5 按花卉种类分组汇总

In [ ]:
summary = df.groupby("species")[["Petal.Ratio", "Sepal.Ratio"]].agg(
    ["mean", "median", "min", "max", "std"]
)
print(summary.round(4))

**说明**

使用一次 `groupby("species")` 加 `.agg([...])` 即同时得到两个新特征的平均值、中位数、最小值、最大值和标准差。

## 提交前检查

1. 重启内核并运行全部代码单元，确认没有依赖旧变量；
2. 将 Notebook 命名为 `20244080228_杨俊兴_hw03.ipynb`；
3. 随作业提交运行生成的 `iris_corrected.csv`。